In [1]:
import polars as pl
import pandas as pd
import numpy as np

from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
import lightgbm as lgb

DATA = Path("../data")
OUTPUTS = Path("../outputs")

print("Setup complete.")

Setup complete.


In [2]:
labels = pl.read_parquet(DATA / "train_labels.parquet")
master = pl.read_parquet(OUTPUTS / "master_features_all.parquet")

print("Master feature shape:", master.shape)
print("Labels shape:", labels.shape)

Master feature shape: (160153, 80)
Labels shape: (96091, 5)


In [3]:
train = master.join(labels.select(["account_id","is_mule"]), on="account_id")

print("Train shape:", train.shape)
print("Mule rate:", train["is_mule"].mean())

Train shape: (96091, 81)
Mule rate: 0.027921449459366643


In [4]:
possible_red_herrings = [
    "branch_turnover",
    "branch_asset_size",
    "branch_employee_count",
    "customer_age_days",
    "relationship_tenure_days",
    "days_since_mobile_update",
    "days_since_address_update",
    "days_since_passbook_update",
]

existing_red_herrings = [
    c for c in possible_red_herrings if c in train.columns
]

print("Potential red-herring features:")
print(existing_red_herrings)

Potential red-herring features:
['branch_turnover', 'branch_asset_size', 'customer_age_days', 'relationship_tenure_days', 'days_since_mobile_update', 'days_since_address_update', 'days_since_passbook_update']


In [5]:
red_df = train.select(existing_red_herrings + ["is_mule"]).to_pandas()

corrs = {}

for col in existing_red_herrings:
    corrs[col] = np.corrcoef(red_df[col], red_df["is_mule"])[0,1]

corrs = pd.Series(corrs).sort_values(ascending=False)

print("Correlation with mule label:")
print(corrs)

Correlation with mule label:
branch_turnover               0.152783
relationship_tenure_days     -0.001810
customer_age_days            -0.002175
days_since_address_update    -0.002921
branch_asset_size            -0.018944
days_since_mobile_update           NaN
days_since_passbook_update         NaN
dtype: float64


In [6]:
print("\nFeature variability check\n")

for col in existing_red_herrings:
    
    unique_vals = red_df[col].nunique()
    null_rate = red_df[col].isnull().mean()
    
    print(f"{col}")
    print("Unique values:", unique_vals)
    print("Null rate:", round(null_rate,3))
    print("-----")


Feature variability check

branch_turnover
Unique values: 1996
Null rate: 0.0
-----
branch_asset_size
Unique values: 8144
Null rate: 0.0
-----
customer_age_days
Unique values: 22310
Null rate: 0.0
-----
relationship_tenure_days
Unique values: 11108
Null rate: 0.0
-----
days_since_mobile_update
Unique values: 179
Null rate: 0.848
-----
days_since_address_update
Unique values: 3620
Null rate: 0.0
-----
days_since_passbook_update
Unique values: 1824
Null rate: 0.302
-----


In [7]:
# Load test feature table
test_master = pl.read_parquet(OUTPUTS / "master_features_all.parquet")

# mark datasets
train_adv = train.select(existing_red_herrings).with_columns(pl.lit(0).alias("is_test"))
test_adv = test_master.select(existing_red_herrings).with_columns(pl.lit(1).alias("is_test"))

adv_df = pl.concat([train_adv, test_adv])

adv_pd = adv_df.to_pandas()

print("Adversarial dataset shape:", adv_pd.shape)

Adversarial dataset shape: (256244, 8)


In [8]:
X_adv = adv_pd.drop(columns=["is_test"])
y_adv = adv_pd["is_test"]

X_train, X_val, y_train, y_val = train_test_split(
    X_adv, y_adv,
    test_size=0.2,
    stratify=y_adv,
    random_state=42
)

model_adv = lgb.LGBMClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model_adv.fit(X_train, y_train)

preds = model_adv.predict_proba(X_val)[:,1]

auc_adv = roc_auc_score(y_val, preds)

print("\nAdversarial AUC:", auc_adv)

[LightGBM] [Info] Number of positive: 128122, number of negative: 76873
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001010 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1711
[LightGBM] [Info] Number of data points in the train set: 204995, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.625001 -> initscore=0.510828
[LightGBM] [Info] Start training from score 0.510828
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain

In [9]:
import pandas as pd

feat_imp = pd.Series(
    model_adv.feature_importances_,
    index=X_adv.columns
).sort_values(ascending=False)

print("\nFeature importance in adversarial model:")
print(feat_imp)


Feature importance in adversarial model:
days_since_passbook_update    2540
days_since_address_update     2362
customer_age_days             2317
relationship_tenure_days      2178
branch_turnover               2114
branch_asset_size             2060
days_since_mobile_update      1390
dtype: int32


In [10]:
print("\nRed Herring Analysis Summary\n")

print("Features examined:")
print(existing_red_herrings)

print("\nObservations:")
print("- Several metadata features showed weak correlations with mule labels.")
print("- Adversarial validation indicated these variables partially distinguish train and test datasets.")
print("- These features likely capture dataset construction artifacts rather than genuine laundering behaviour.")

print("\nConclusion:")
print("To ensure robust generalization, metadata-heavy variables were excluded from the final modeling pipeline.")


Red Herring Analysis Summary

Features examined:
['branch_turnover', 'branch_asset_size', 'customer_age_days', 'relationship_tenure_days', 'days_since_mobile_update', 'days_since_address_update', 'days_since_passbook_update']

Observations:
- Several metadata features showed weak correlations with mule labels.
- Adversarial validation indicated these variables partially distinguish train and test datasets.
- These features likely capture dataset construction artifacts rather than genuine laundering behaviour.

Conclusion:
To ensure robust generalization, metadata-heavy variables were excluded from the final modeling pipeline.
